<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자기지도 학습 · 멀티모달 · 09. Flamingo: frozen LM에 시각 cross-attention

## Flamingo: a Visual Language Model for Few-Shot Learning

- **원 논문:** [Flamingo: a Visual Language Model for Few-Shot Learning](https://arxiv.org/abs/2204.14198)
- **저자 / 발표:** Jean-Baptiste Alayrac et al. · NeurIPS (2022)
- **난이도 / 예상 시간:** 고급 · 약 90분
- **선수 지식:** cross-attention, language modeling, few-shot prompting
- **로컬 학습 데이터:** `data/field_curriculum/multimodal_pairs.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** Perceiver Resampler와 tanh-gated cross-attention으로 image token을 text stream에 주입합니다.

**축소하거나 생략한 부분:** Chinchilla 기반 3B–80B 모델·43M interleaved documents 대신 8×8 image-caption pair를 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2.1 및 Figure 3 | Perceiver Resampler의 고정 visual tokens | latent shape |
| §2.2 및 Figure 4 | frozen LM 사이 gated cross-attention | gate=0 identity |
| Eq. (1) | interleaved next-token likelihood | causal CE |
| §2.3 | vision/LM freeze, bridge만 학습 | trainable parameter audit·conditioning gap |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** visual [B,Nᵥ,D] + text [B,T,D] → conditioned hidden [B,T,D]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(17)

DATA_PATH = Path("data/field_curriculum/multimodal_pairs.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
images = torch.tensor(raw["images"], dtype=torch.float32)
tokens = torch.tensor(raw["tokens"], dtype=torch.long)
class_ids = torch.tensor(raw["class_ids"], dtype=torch.long)
sequences = torch.tensor(raw["sequences"], dtype=torch.float32)
train_idx = torch.tensor(raw["train_idx"], dtype=torch.long)
test_idx = torch.tensor(raw["test_idx"], dtype=torch.long)
assert len(images) == len(tokens) == len(class_ids) == len(sequences)
assert images.shape[1:] == (1, 8, 8) and tokens.ndim == 2 and sequences.ndim == 3
print(
    f"pairs={len(images)}, classes={class_ids.unique().numel()}, "
    f"image={tuple(images.shape[1:])}, tokens={tokens.shape[1]}"
)
print(ACCELERATOR.summary())
print("dataset/index preprocessing stays on CPU, while neural training uses DEVICE")

## 포트폴리오 해설: 표현, 목적함수, 업데이트

$$p(y_\ell\mid y_{<\ell},x)=\operatorname{softmax}\!\left(W(h_\ell+\tanh(g)
\operatorname{XAttn}(h_\ell,V_x))\right)$$

- **기호 정의:** h는 frozen LM state, V_x는 resampled visual token, g는 학습 gate입니다.
- **수식의 역할:** 고정 LM 사이에 0으로 초기화한 gated cross-attention을 삽입해 시각 조건을 줍니다.
- **구현 이유:** representation 붕괴·modality 누설·잘못된 positive 연결을
  코드에서 확인할 수 있도록 핵심 수식을 view 준비와 loss 경계로 분리했습니다.
- **Task/코드 대응:** Task P의 `prepare_views`, 논문 전용 `nn.Module`의
  `forward`, `compute_loss`, `training_step`이 §2.1 및 Figure 3을 구현합니다.
- **입출력 shape:** image [B, 1, 8, 8], token [B, L] → visual latent [B, M, D] → LM logit
- **평가:** next-token CE와 correct/shuffled image conditioning gap
- **원문 대비 한계:** Chinchilla 기반 3B–80B 모델·43M interleaved documents 대신 8×8 image-caption pair를 사용합니다.

구현은 `로컬 데이터 통계 → view/modality 준비 → model → objective/update → 평가`
순서입니다. 이 순서는 데이터 누설과 stop-gradient 경계를 코드에서 찾기 쉽게 합니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2.1 및 Figure 3
- **이 셀의 책임:** Perceiver Resampler의 고정 visual tokens
- **Tensor shape 계약:** visual [B,Nᵥ,D] + text [B,T,D] → conditioned hidden [B,T,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** latent shape. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class ExplicitCrossAttention(nn.Module):
    def __init__(self, dim=16, heads=2):
        super().__init__()
        if dim % heads:
            raise ValueError("dim must be divisible by heads")
        self.heads = heads
        self.head_dim = dim // heads
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.output = nn.Linear(dim, dim)

    def split_heads(self, tensor):
        batch, tokens, dim = tensor.shape
        return tensor.reshape(batch, tokens, self.heads, self.head_dim).transpose(1, 2)

    def merge_heads(self, tensor):
        batch, _, tokens, _ = tensor.shape
        return tensor.transpose(1, 2).reshape(
            batch,
            tokens,
            self.heads * self.head_dim,
        )

    def forward(self, query, key, value):
        query_heads = self.split_heads(self.query(query))
        key_heads = self.split_heads(self.key(key))
        value_heads = self.split_heads(self.value(value))
        scores = query_heads @ key_heads.transpose(-2, -1)
        weights = (scores / self.head_dim**0.5).softmax(dim=-1)
        attended = weights @ value_heads
        return self.output(self.merge_heads(attended)), weights


class PerceiverResampler(nn.Module):
    def __init__(self, dim=16, n_latents=4):
        super().__init__()
        self.patch = nn.Linear(4, dim)
        self.latents = nn.Parameter(torch.randn(1, n_latents, dim) * 0.02)
        self.attn = ExplicitCrossAttention(dim, heads=2)

    def forward(self, image):
        patch = (
            image.unfold(2, 2, 2)
            .unfold(3, 2, 2)
            .permute(0, 2, 3, 1, 4, 5)
            .reshape(len(image), 16, 4)
        )
        kv = self.patch(patch)
        q = self.latents.expand(len(image), -1, -1)
        return self.attn(q, kv, kv)[0]


resampler = PerceiverResampler()
visual = resampler(images[:8])
assert visual.shape == (8, 4, 16)

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2.2 및 Figure 4
- **이 셀의 책임:** frozen LM 사이 gated cross-attention
- **Tensor shape 계약:** visual [B,Nᵥ,D] + text [B,T,D] → conditioned hidden [B,T,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** gate=0 identity. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class GatedCrossAttention(nn.Module):
    def __init__(self, dim=16):
        super().__init__()
        self.attention_norm = nn.LayerNorm(dim)
        self.attn = ExplicitCrossAttention(dim, heads=2)
        self.gate = nn.Parameter(torch.zeros(()))
        self.ff = nn.Sequential(
            nn.LayerNorm(dim), nn.Linear(dim, 32), nn.GELU(), nn.Linear(32, dim)
        )
        self.ff_gate = nn.Parameter(torch.zeros(()))

    def forward(self, text, visual):
        attended = self.attn(self.attention_norm(text), visual, visual)[0]
        h = text + torch.tanh(self.gate) * attended
        return h + torch.tanh(self.ff_gate) * self.ff(h)


cross = GatedCrossAttention()
probe = torch.randn(8, 5, 16)
assert torch.allclose(cross(probe, visual), probe, atol=1e-7)

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §2.1 및 Figure 3 / §2.2 및 Figure 4 / Eq. (1)
- **이 셀의 책임:** Perceiver Resampler의 고정 visual tokens / frozen LM 사이 gated cross-attention / interleaved next-token likelihood
- **Tensor shape 계약:** visual [B,Nᵥ,D] + text [B,T,D] → conditioned hidden [B,T,D]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** latent shape / gate=0 identity / causal CE. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
vocab = int(tokens.max()) + 1
token_backbone = nn.Embedding(vocab, 16)
token_backbone.weight.requires_grad_(False)
head = nn.Linear(16, vocab)
token_backbone, resampler, cross, head = ACCELERATOR.move(
    token_backbone, resampler, cross, head
)
parameters = [*resampler.parameters(), *cross.parameters(), *head.parameters()]
optimizer = torch.optim.Adam(parameters, lr=0.012)
idx = train_idx[:96]
losses = []
train_images, train_tokens = ACCELERATOR.move(images[idx], tokens[idx])
for _ in range(140):
    t = train_tokens
    text = token_backbone(t[:, :-1])
    visual = resampler(train_images)
    logits = head(cross(text, visual))
    loss = F.cross_entropy(logits.reshape(-1, vocab), t[:, 1:].reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
assert (
    losses[-1] < losses[0]
    and token_backbone.weight.grad is None
    and abs(float(torch.tanh(cross.gate))) > 1e-4
)

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** Eq. (1) / §2.3
- **이 셀의 책임:** interleaved next-token likelihood / vision/LM freeze, bridge만 학습
- **Tensor shape 계약:** visual [B,Nᵥ,D] + text [B,T,D] → conditioned hidden [B,T,D]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** causal CE / trainable parameter audit·conditioning gap. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def conditioned_loss(image, text):
    with torch.no_grad():
        h = token_backbone(text[:, :-1])
        logits = head(cross(h, resampler(image)))
        return float(
            F.cross_entropy(logits.reshape(-1, vocab), text[:, 1:].reshape(-1))
            .detach()
            .cpu()
        )


x, t = ACCELERATOR.move(images[test_idx], tokens[test_idx])
correct = conditioned_loss(x, t)
shuffled = conditioned_loss(x.roll(1, 0), t)
assert np.isfinite([correct, shuffled]).all()
plt.plot(losses)
plt.title("Flamingo bridge LM loss")
plt.xlabel("step")
plt.show()
print(
    {
        "correct_image_loss": round(correct, 3),
        "shuffled_image_loss": round(shuffled, 3),
        "visual_conditioning_gap": round(shuffled - correct, 3),
    }
)

### Task P1 · modality/view 준비와 핵심 모델

image·text·sequence의 shape를 먼저 고정하고 augmentation 또는 modality
경계를 명시한 뒤 논문 전용 `nn.Module`을 구성합니다.

### 구현 단계 6 · 핵심 연산·모델

- **원문 위치:** §2.1 및 Figure 3 / §2.2 및 Figure 4 / Eq. (1) / §2.3
- **이 셀의 책임:** Perceiver Resampler의 고정 visual tokens / frozen LM 사이 gated cross-attention / interleaved next-token likelihood / vision/LM freeze, bridge만 학습
- **Tensor shape 계약:** visual [B,Nᵥ,D] + text [B,T,D] → conditioned hidden [B,T,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** latent shape / gate=0 identity / causal CE / trainable parameter audit·conditioning gap. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 16
    visual_tokens: int = 4
    learning_rate: float = 0.02
    steps: int = 14


def prepare_views(image_batch: Tensor, text_batch: Tensor) -> tuple[Tensor, Tensor]:
    return image_batch, text_batch


class FlamingoPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.vocabulary = int(tokens.max()) + 1
        self.visual_projection = nn.Linear(4, config.hidden_dim)
        self.resampler_queries = nn.Parameter(
            torch.randn(config.visual_tokens, config.hidden_dim) * 0.02
        )
        self.token_embedding = nn.Embedding(self.vocabulary, config.hidden_dim)
        self.language_model = nn.GRU(
            config.hidden_dim, config.hidden_dim, batch_first=True
        )
        self.resampler_attention = ExplicitCrossAttention(
            config.hidden_dim,
            heads=2,
        )
        self.gated_cross_attention = ExplicitCrossAttention(
            config.hidden_dim,
            heads=2,
        )
        self.gate = nn.Parameter(torch.tensor(0.0))
        self.token_head = nn.Linear(config.hidden_dim, self.vocabulary)
        frozen_modules = (
            self.visual_projection,
            self.token_embedding,
            self.language_model,
            self.token_head,
        )
        for module in frozen_modules:
            for parameter in module.parameters():
                parameter.requires_grad_(False)
        trainable_names = [
            name
            for name, parameter in self.named_parameters()
            if parameter.requires_grad
        ]
        allowed_prefixes = (
            "resampler_queries",
            "resampler_attention",
            "gated_cross_attention",
            "gate",
        )
        assert trainable_names
        assert all(name.startswith(allowed_prefixes) for name in trainable_names)

    def resample_visual(self, image_batch: Tensor) -> Tensor:
        patch = image_batch.unfold(2, 2, 2).unfold(3, 2, 2)
        patch = patch.permute(0, 2, 3, 1, 4, 5).reshape(len(image_batch), 16, 4)
        visual = self.visual_projection(patch)
        queries = self.resampler_queries[None].expand(len(image_batch), -1, -1)
        latent, _ = self.resampler_attention(queries, visual, visual)
        return latent

    def forward(self, image_batch: Tensor, text_batch: Tensor) -> Tensor:
        visual = self.resample_visual(image_batch)
        token_hidden = self.token_embedding(text_batch[:, :-1])
        language_hidden, _ = self.language_model(token_hidden)
        conditioned, _ = self.gated_cross_attention(
            language_hidden,
            visual,
            visual,
        )
        fused = language_hidden + torch.tanh(self.gate) * conditioned
        return self.token_head(fused)

    def compute_loss(self, logits: Tensor, text_batch: Tensor) -> Tensor:
        return F.cross_entropy(
            logits.reshape(-1, self.vocabulary),
            text_batch[:, 1:].reshape(-1),
        )

    def training_step(self, image_batch: Tensor, text_batch: Tensor) -> Tensor:
        return self.compute_loss(self(image_batch, text_batch), text_batch)

### Task P2 · objective, stop-gradient, update

positive/negative, online/target 또는 masked/visible 경계를 확인하면서
`fit_portfolio_model`의 gradient와 EMA update 순서를 추적합니다.

### 구현 단계 7 · 목적함수·학습 update

- **원문 위치:** Eq. (1) / §2.3
- **이 셀의 책임:** interleaved next-token likelihood / vision/LM freeze, bridge만 학습
- **Tensor shape 계약:** visual [B,Nᵥ,D] + text [B,T,D] → conditioned hidden [B,T,D]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** causal CE / trainable parameter audit·conditioning gap. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(
    model: FlamingoPortfolioModel,
    image_batch: Tensor,
    text_batch: Tensor,
) -> list[float]:
    trainable = [
        (name, parameter)
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    ]
    allowed_prefixes = (
        "resampler_queries",
        "resampler_attention",
        "gated_cross_attention",
        "gate",
    )
    assert trainable
    assert all(name.startswith(allowed_prefixes) for name, _ in trainable)
    optimizer = torch.optim.Adam(
        [parameter for _, parameter in trainable],
        lr=model.config.learning_rate,
    )
    history = []
    for _ in range(model.config.steps):
        loss = model.training_step(image_batch, text_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 표현 품질 평가

학습 loss와 retrieval·reconstruction·conditioning 지표를 분리해 계산합니다.
작은 로컬 pair의 수치는 원 논문의 downstream benchmark와 직접 비교하지 않습니다.

### 구현 단계 8 · 평가·완료 증거

- **원문 위치:** Eq. (1) / §2.3
- **이 셀의 책임:** interleaved next-token likelihood / vision/LM freeze, bridge만 학습
- **Tensor shape 계약:** visual [B,Nᵥ,D] + text [B,T,D] → conditioned hidden [B,T,D]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** causal CE / trainable parameter audit·conditioning gap. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: FlamingoPortfolioModel,
    image_batch: Tensor,
    text_batch: Tensor,
) -> dict[str, float]:
    with torch.no_grad():
        correct = model.training_step(image_batch, text_batch)
        shuffled = model.training_step(image_batch.roll(1, 0), text_batch)
    return {
        "language_loss": float(correct.cpu()),
        "conditioning_gap": float((shuffled - correct).cpu()),
    }


portfolio_train_index = train_idx[:48]
portfolio_test_index = test_idx
assert not torch.isin(portfolio_train_index, portfolio_test_index).any()
portfolio_train_images, portfolio_train_tokens = prepare_views(
    images[portfolio_train_index],
    tokens[portfolio_train_index],
)
portfolio_test_images, portfolio_test_tokens = prepare_views(
    images[portfolio_test_index],
    tokens[portfolio_test_index],
)
(
    portfolio_train_images,
    portfolio_train_tokens,
    portfolio_test_images,
    portfolio_test_tokens,
) = ACCELERATOR.move(
    portfolio_train_images,
    portfolio_train_tokens,
    portfolio_test_images,
    portfolio_test_tokens,
)
portfolio_model = ACCELERATOR.move(FlamingoPortfolioModel(PortfolioConfig()))
portfolio_history = fit_portfolio_model(
    portfolio_model,
    portfolio_train_images,
    portfolio_train_tokens,
)
portfolio_metrics = evaluate_portfolio_model(
    portfolio_model,
    portfolio_test_images,
    portfolio_test_tokens,
)
assert np.isfinite(portfolio_history).all()
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> Chinchilla 기반 3B–80B 모델·43M interleaved documents 대신 8×8 image-caption pair를 사용합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.